In [88]:
import keras
import tensorflow as tf
import mlflow
from mlflow.models import infer_signature
from hyperopt import fmin, tpe, hp, Trials, STATUS_OK
from sklearn.model_selection import train_test_split
from sklearn.datasets import load_wine
from sklearn.metrics import mean_squared_error
import os



In [51]:
## load the dataset
data=pd.read_csv(
    "https://raw.githubusercontent.com/mlflow/mlflow/master/tests/datasets/winequality-white.csv",
    sep=";",
)
data

,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,alcohol,quality
0,7.0,0.27,0.36,20.7,0.045,45.0,170.0,1.00100,3.00,0.45,8.8,6
1,6.3,0.30,0.34,1.6,0.049,14.0,132.0,0.99400,3.30,0.49,9.5,6
2,8.1,0.28,0.40,6.9,0.050,30.0,97.0,0.99510,3.26,0.44,10.1,6
3,7.2,0.23,0.32,8.5,0.058,47.0,186.0,0.99560,3.19,0.40,9.9,6
4,7.2,0.23,0.32,8.5,0.058,47.0,186.0,0.99560,3.19,0.40,9.9,6
...,...,...,...,...,...,...,...,...,...,...,...,...
4893,6.2,0.21,0.29,1.6,0.039,24.0,92.0,0.99114,3.27,0.50,11.2,6
4894,6.6,0.32,0.36,8.0,0.047,57.0,168.0,0.99490,3.15,0.46,9.6,5
4895,6.5,0.24,0.19,1.2,0.041,30.0,111.0,0.99254,2.99,0.46,9.4,6
4896,5.5,0.29,0.30,1.1,0.022,20.0,110.0,0.98869,3.34,0.38,12.8,7


In [79]:
train, test= train_test_split(data, test_size=0.2, random_state=42)

train


,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,alcohol,quality
4665,7.3,0.17,0.36,8.20,0.028,44.0,111.0,0.99272,3.14,0.41,12.4,6
1943,6.3,0.25,0.44,11.60,0.041,48.0,195.0,0.99680,3.18,0.52,9.5,5
3399,5.6,0.32,0.33,7.40,0.037,25.0,95.0,0.99268,3.25,0.49,11.1,6
843,6.9,0.19,0.35,1.70,0.036,33.0,101.0,0.99315,3.21,0.54,10.8,7
2580,7.7,0.30,0.26,18.95,0.053,36.0,174.0,0.99976,3.20,0.50,10.4,5
...,...,...,...,...,...,...,...,...,...,...,...,...
4426,6.2,0.21,0.52,6.50,0.047,28.0,123.0,0.99418,3.22,0.49,9.9,6
466,7.0,0.14,0.32,9.00,0.039,54.0,141.0,0.99560,3.22,0.43,9.4,6
3092,7.6,0.27,0.52,3.20,0.043,28.0,152.0,0.99129,3.02,0.53,11.4,6
3772,6.3,0.24,0.29,13.70,0.035,53.0,134.0,0.99567,3.17,0.38,10.6,6


In [80]:
train[['quality']].values.ravel()

array([6, 5, 6, ..., 6, 6, 8], shape=(3918,))

In [94]:
train_x= train.drop('quality', axis=1).values
train_x

array([[ 7.3 ,  0.17,  0.36, ...,  3.14,  0.41, 12.4 ],
       [ 6.3 ,  0.25,  0.44, ...,  3.18,  0.52,  9.5 ],
       [ 5.6 ,  0.32,  0.33, ...,  3.25,  0.49, 11.1 ],
       ...,
       [ 7.6 ,  0.27,  0.52, ...,  3.02,  0.53, 11.4 ],
       [ 6.3 ,  0.24,  0.29, ...,  3.17,  0.38, 10.6 ],
       [ 8.1 ,  0.27,  0.35, ...,  3.22,  0.63, 10.4 ]], shape=(3918, 11))

In [81]:
train_x= train.drop('quality', axis=1).values
train_y= train[['quality']].values.ravel()

# test data
test_x= test.drop('quality', axis=1).values
test_y= test[['quality']].values.ravel()

# spliting data into train and validation
train_x, val_x, train_y, val_y= train_test_split(train_x, 
                                                 train_y,
                                                 test_size=0.2, 
                                                 random_state=42)
signature= infer_signature(train_x, train_y)


In [56]:
train_x.shape[1]

11

In [90]:
# ANN model
from hyperopt import STATUS_OK

def train_model(params, epochs, train_x, train_y, val_x, val_y, test_x, test_y):
    
    mean= train_x.mean(axis=0)
    var= train_x.var(axis=0)
    
    model= keras.Sequential(
        [
            keras.Input([train_x.shape[1]]),
            keras.layers.Normalization(mean= mean, variance= var),
            keras.layers.Dense(64, activation= 'relu'),
            keras.layers.Dense(32, activation= 'relu'),
            keras.layers.Dense(1)
        ]
    )
   
    
    model.compile(
        optimizer= keras.optimizers.SGD(learning_rate=params['learning_rate'],
                                         momentum= params["momentum"]),
        loss='mse',
        metrics=['mse']
    )
    
    with mlflow.start_run(nested= True):
        model.fit(
        train_x, train_y,
        validation_data=(val_x, val_y),
        epochs=epochs,
        batch_size=64,
        verbose=True)
        
        eval_result= model.evaluate(val_x, val_y, batch_size=64)
        
        eval_rmse= eval_result[1]
        
        mlflow.log_metric("eval_rmse", eval_rmse)
        
        # log model
        mlflow.tensorflow.log_model(model, "model", signature= signature)
        
        return {"loss": eval_rmse, 'status':STATUS_OK, 'model': model}
        
    

In [83]:
def objective(params):
    return train_model(params, epochs=10, 
                       train_x= train_x, 
                       train_y= train_y, 
                       val_x= val_x, 
                       val_y= val_y,
                       test_x= test_x,
                       test_y= test_y)
    return result

In [85]:
space= {
    'learning_rate': hp.uniform('learning_rate', 0.00001, 0.1),
    'momentum': hp.uniform('momentum', 0.0, 0.1)
}

In [86]:
# Disable the parts of autologging that are causing the schema crash
mlflow.tensorflow.autolog(
    log_models=False,           # Stops it from trying to save the .h5/.keras file
    log_model_signatures=False,  # Stops the TensorSpec validation error
    log_datasets=False           # Prevents issues with Pandas/Numpy conversion
)

In [91]:
mlflow.set_tracking_uri("http://127.0.0.1:5000")

mlflow.set_experiment("Wine_Quality_Local_v2")
with mlflow.start_run():
    trials = Trials()
    best = fmin(
        fn=objective,
        space=space,
        algo=tpe.suggest,
        max_evals=10, 
        trials=trials
    )
    
    # fetch the details of the best run
    best_run= sorted(trials.results, key= lambda x: x["loss"])[0]
    
    # log the best model parameters and metrics
    mlflow.log_params(best)
    mlflow.log_metric("best_eval_rmse", best_run["loss"])
    mlflow.tensorflow.log_model(best_run["model"], "model", signature=signature)
    
    # print the best hyperparameters and evaluation metric
    print("Best Hyperparameters:", best)    
    print("Best Evaluation RMSE:", best_run["loss"])
    
    
    


  0%|          | 0/10 [00:00<?, ?trial/s, best loss=?]

Epoch 1/10                                            

 1/49 ━━━━━━━━━━━━━━━━━━━━ 46s 970ms/step - loss: 38.6304 - mse: 38.6304
13/49 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 14.0323 - mse: 14.0323   
28/49 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 9.2046 - mse: 9.2046  
43/49 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 7.0873 - mse: 7.0873
  0%|          | 0/10 [00:01<?, ?trial/s, best loss=?]

49/49 ━━━━━━━━━━━━━━━━━━━━ 2s 20ms/step - loss: 2.4545 - mse: 2.4545 - val_loss: 0.6655 - val_mse: 0.6655

Epoch 2/10                                            

 1/49 ━━━━━━━━━━━━━━━━━━━━ 2s 51ms/step - loss: 0.6652 - mse: 0.6652
20/49 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.6110 - mse: 0.6110 
37/49 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.6015 - mse: 0.6015
  0%|          | 0/10 [00:02<?, ?trial/s, best loss=?]

49/49 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - loss: 0.5952 - mse: 0.5952 - val_loss: 0.4833 - val_mse: 0.4833

Epoch 3/10                                            

 1/49 ━━━━━━━━━━━━━━━━━━━━ 2s 45ms/step - loss: 0.5155 - mse: 0.5155
20/49 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.4983 - mse: 0.4983 
39/49 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.5089 - mse: 0.5089
  0%|          | 0/10 [00:03<?, ?trial/s, best loss=?]

49/49 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - loss: 0.5240 - mse: 0.5240 - val_loss: 0.4748 - val_mse: 0.4748

Epoch 4/10                                            

 1/49 ━━━━━━━━━━━━━━━━━━━━ 2s 47ms/step - loss: 0.2888 - mse: 0.2888
20/49 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.4923 - mse: 0.4923 
39/49 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.4935 - mse: 0.4935
49/49 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 0.5070 - mse: 0.5070 - val_loss: 0.5581 - val_mse: 0.5581

Epoch 5/10                                            

 1/49 ━━━━━━━━━━━━━━━━━━━━ 2s 53ms/step - loss: 0.4446 - mse: 0.4446
19/49 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.5117 - mse: 0.5117 
39/49 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.5103 - mse: 0.5103
  0%|          | 0/10 [00:04<?, ?trial/s, best loss=?]

...

49/49 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step - loss: 0.5371 - mse: 0.5371 - val_loss: 0.4618 - val_mse: 0.4618

Epoch 4/10                                                                      

 1/49 ━━━━━━━━━━━━━━━━━━━━ 2s 45ms/step - loss: 0.5713 - mse: 0.5713
20/49 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.5179 - mse: 0.5179 
40/49 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.5001 - mse: 0.5001
49/49 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 0.5173 - mse: 0.5173 - val_loss: 0.4873 - val_mse: 0.4873

Epoch 5/10                                                                      

 1/49 ━━━━━━━━━━━━━━━━━━━━ 2s 53ms/step - loss: 0.5221 - mse: 0.5221
19/49 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.5524 - mse: 0.5524 
37/49 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.5263 - mse: 0.5263
49/49 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 0.5044 - mse: 0.5044 - val_loss: 0.4898 - val_mse: 0.4898

Epoch 6/10                                                                      

 1/49 ━━━━━━━━━━━━━━━━━

2026/05/01 12:16:13 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.



🏃 View run upbeat-wren-263 at: http://127.0.0.1:5000/#/experiments/4/runs/f8cc2ed347e04652b219677d6cf226cb

🧪 View experiment at: http://127.0.0.1:5000/#/experiments/4                    

100%|██████████| 10/10 [03:50<00:00, 23.03s/trial, best loss: 0.4538724422454834]


2026/05/01 12:16:29 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


Best Hyperparameters: {'learning_rate': np.float64(0.09793152182150718), 'momentum': np.float64(0.08697887382780649)}
Best Evaluation RMSE: 0.4538724422454834
🏃 View run handsome-asp-465 at: http://127.0.0.1:5000/#/experiments/4/runs/0016f57cade745619a2c5f096f5a5076
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/4
